# Run demanding OpenMM notebooks non-interactively on an HPC

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

A non-interactive job can be configured and submitted to the queue as follows:

1. All files needed for this hands-on are already present in the `03_noninteractive` directory
   of the tutorials you downloaded to `${VSC_SCRATCH}/phystack/famd/` on the HPC:

   - `01_noninteractive_notebook_on_hpc.ipynb`
   - `alanine-dipeptide.pdb`
   - `job_openmm_vsc_cpu.sh` (for the UGent VSC Tier-2 clusters)
   - `job_openmm_vsc_gpu.sh` (for the UGent VSC Tier-2 clusters with GPUs)

   Open a virtual terminal and `cd` into this directory before you continue.

1. The files starting with `job_` are job scripts that define the calculation that needs to be performed on a cluster.
   We provide basic job scripts that may require some additional modifications to better suit your needs.
   If the cluster you intend to use has GPUs, start from `job_openmm_vsc_gpu.sh`, or use `job_openmm_vsc_cpu.sh` otherwise.
   Both scripts use only software modules installed on the cluster,
   and not the PhyStack Python environment in `${VSC_SCRATCH}/phystack/`.

1. When you reuse a job script for another notebook, you should edit it and change `01_noninteractive_notebook_on_hpc.ipynb` to the name of your notebook.
   You can also change the `SBATCH` arguments to suit your needs:

   - Number of CPU cores is set by `--cpus-per-task`.
   - Number of GPUs is set by `--gpus-per-node`. (Not more than 1, because OpenMM runs a single simulation on a single GPU.)
   - Maximum wall time is set by `--time`. (Your job will be killed in case your calculation does not stop in time.)
   - Maximum memory usage is set by `--mem`. (Your job will be killed when it uses more.)

   The `sbatch` program supports many other options to control the execution of the job.
   Consult the [`sbatch` documentation](https://slurm.schedmd.com/sbatch.html) for more details.

1. Select the cluster on which the job should run.
   The default cluster is `skiddo` when submitting jobs.
   To debug a job script, it is recommended to test on the Donphan cluster first.
   Once it is working as expected, you can switch to a production cluster.
   To use another cluster, run the following command before calling `sbatch`:

   ```bash
   module swap cluster/donphan
   ```

   To see the available clusters:

   ```bash
   module av cluster/
   ```

   A job script can only run on a cluster where the modules it loads (`ml load ...`) are installed.
   After switching to a cluster, you can check this as follows:

   ```bash
   module avail OpenMM jupyter-server
   ```

1. Finally, when your job script is ready, you can submit it by entering the command `sbatch job_openmm_vsc_cpu.sh` in the virtual terminal.
   This will put your job on the queue.
   As soon as resources are available to run your job, it will be executed.

1. You can check the status of your job by entering the `squeue` command in the virtual terminal.
   You can also check your job queue on [login.hpc.ugent.be](https://login.hpc.ugent.be).
   Note that in a virtual terminal, you only get to see your jobs running on the current cluster.
   Also for this, you need to switch to the right cluster, e.g. with `module swap cluster/...`.

1. While the job is running, the outputs of the notebook cells are not visible yet.
   (The file `slurm-<jobid>.out` only contains the output of `nbconvert`, not of the notebook.)
   To follow the progress of the simulation, this notebook writes a file `progress3.txt`,
   which you can monitor in the terminal with `tail -f progress3.txt`. (Press `Ctrl+C` to stop.)

1. Once the job has completed, a copy of this notebook is created with all the outputs
   (`01_noninteractive_notebook_on_hpc.nbconvert.ipynb`), and also all other output files can be found in the same directory.
   Because the job scripts use the `--allow-errors` option, the job will also complete when a cell raises an exception.
   Always check the output notebook for error messages.

**Notes for the GPU job script:**

- The GPU job script sets a variable `OPENMM_DEFAULT_PLATFORM=CUDA`, which tells OpenMM to use GPUs.
  The CPU job script sets `OPENMM_DEFAULT_PLATFORM=CPU`.
  The platform actually used by OpenMM is printed in the output notebook.

- The GPU clusters `accelgor`, `joltik` and `litleo` only give a job access to a GPU if it is requested in the job script with `--gpus-per-node`.

- Donphan is a special case: GPUs cannot be requested,
  so you must remove the `--gpus-per-node` line from the job script.
  A job running on Donphan can always try to use its GPU,
  which may or may not work, depending on who else is using it.

- When running on a GPU, OpenMM barely uses the CPU, so there is no need to request more than two CPU cores.

- The GPU speedup for this notebook (on Joltik) is about a factor of 30.
  Compare the `Speed (ns/day)` column in the output of a CPU and a GPU job to verify this.

In [ ]:
# Import all the modules we need.
from sys import stdout

import openmm as mm
from openmm import app, unit

The following code was taken from [../02_alanine_dipeptide/01_force_fields.ipynb](../02_alanine_dipeptide/01_force_fields.ipynb), section 3.

In [ ]:
pdb = app.PDBFile("alanine-dipeptide.pdb")
modeller = app.Modeller(pdb.topology, pdb.positions)
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
# The model argument selects the geometry of the water molecules to be added.
# TIP3P-FB is a three-site model, just like TIP3P.
modeller.addSolvent(forcefield, model="tip3p", padding=1.2 * unit.nanometer)
print(modeller.topology)
# Write a PDB file to provide a topology of the solvated
# system, e.g. for a later analysis of the trajectory with MDTraj.
with open("init3.pdb", "w") as outfile:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, outfile)

# The modeller builds a periodic box with the solute and solvent molecules.
# PME is the method to compute long-range electrostatic interactions in
# periodic systems.
system = forcefield.createSystem(modeller.topology, nonbondedMethod=app.PME, constraints=app.HBonds)
temperature = 300 * unit.kelvin
pressure = 1 * unit.bar
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, 2 * unit.femtoseconds)
system.addForce(mm.MonteCarloBarostat(pressure, temperature))
simulation = app.Simulation(modeller.topology, system, integrator)
print("Platform:", simulation.context.getPlatform().getName())
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy()
simulation.reporters.append(app.DCDReporter("traj3.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, speed=True, elapsedTime=True)
)
# Outputs of the notebook are only visible after it has completed.
# The following file can be monitored while the simulation is running.
simulation.reporters.append(
    app.StateDataReporter(
        "progress3.txt",
        1000,
        progress=True,
        remainingTime=True,
        speed=True,
        totalSteps=100000,
    )
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars3.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(100000)